In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(override=True)

config = {
    "uri":      os.environ["NEO4J_URI"],
    "user":     os.environ["NEO4J_USER"],
    "password": os.environ["NEO4J_PASSWORD"],
    "db":       os.environ["NEO4J_DB"],
}

print(config["db"])  # verifica que sea proyecto-2

proyecto-2


In [2]:
driver = GraphDatabase.driver(config["uri"], auth=(config["user"], config["password"]))

# Crear el database si no existe
with driver.session(database="system") as s:
    s.run(f"CREATE DATABASE `{config['db']}` IF NOT EXISTS")
    print(f"Database '{config['db']}' listo")

# Verificar conexión al nuevo database
with driver.session(database=config["db"]) as s:
    result = s.run("RETURN 1")
    if result.single()[0] == 1:
        print(f"Conexión exitosa a '{config['db']}'")

Database 'proyecto-2' listo
Conexión exitosa a 'proyecto-2'


In [3]:
import pandas as pd

nodes_df = pd.read_csv("data/processed/metro_nodes.csv")
edges_df = pd.read_csv("data/processed/metro_edges.csv")
afluencia_df = pd.read_csv("data/processed/afluencia_diaria.csv")

print(f"Estaciones:  {len(nodes_df)}")
print(f"Conexiones:  {len(edges_df)}")
print(f"Afluencia:   {len(afluencia_df)}")

Estaciones:  163
Conexiones:  48
Afluencia:   71370


In [4]:
with driver.session(database=config["db"]) as s:
    count = s.run("MATCH (e:Estacion) RETURN count(e) AS n").single()["n"]

if count >= len(nodes_df):
    print(f"✓ Estaciones ya importadas ({count}), skip")
else:
    rows = nodes_df.rename(columns={"AÑO": "ANIO"}).to_dict("records")
    with driver.session(database=config["db"]) as s:
        s.run("CREATE CONSTRAINT estacion_id IF NOT EXISTS FOR (e:Estacion) REQUIRE e.cve_est IS UNIQUE")
        s.run("""
            UNWIND $rows AS row
            MERGE (e:Estacion {cve_est: row.CVE_EST})
            SET e.nombre   = row.NOMBRE,
                e.linea    = row.LINEA,
                e.sistema  = row.SISTEMA,
                e.tipo     = row.TIPO,
                e.alcaldia = row.ALCALDIAS,
                e.anio     = row.ANIO,
                e.lon      = row.lon,
                e.lat      = row.lat
        """, rows=rows)
    print(f"✓ {len(rows)} estaciones importadas")

✓ Estaciones ya importadas (163), skip


In [5]:
with driver.session(database=config["db"]) as s:
    count = s.run("MATCH ()-[r:CONECTA]->() RETURN count(r) AS n").single()["n"]

if count >= len(edges_df):
    print(f"✓ Conexiones ya importadas ({count}), skip")
else:
    with driver.session(database=config["db"]) as s:
        s.run("""
            UNWIND $rows AS row
            MATCH (o:Estacion {nombre: row.origen})
            MATCH (d:Estacion {nombre: row.destino})
            MERGE (o)-[r:CONECTA]->(d)
            SET r.linea              = row.linea,
                r.sistema            = row.sistema,
                r.distancia_track_m  = row.distancia_track_m,
                r.distancia_track_km = row.distancia_track_km,
                r.distancia_recta_m  = row.distancia_recta_m,
                r.distancia_recta_km = row.distancia_recta_km
        """, rows=edges_df.to_dict("records"))
    print(f"✓ {len(edges_df)} conexiones importadas")

✓ Conexiones ya importadas (48), skip


In [ ]:
with driver.session(database=config["db"]) as s:
    count = s.run("MATCH (a:Afluencia) RETURN count(a) AS n").single()["n"]

if count >= len(afluencia_df):
    print(f"✓ Afluencia ya importada ({count} registros), skip")
else:
    # Borrar registros previos si el conteo no coincide (clave incorrecta antes)
    if count > 0:
        print(f"  Borrando {count} registros con clave incorrecta...")
        with driver.session(database=config["db"]) as s:
            s.run("MATCH (a:Afluencia) DETACH DELETE a")

    rows = afluencia_df.to_dict("records")
    total = len(rows)
    batch_size = 1000
    with driver.session(database=config["db"]) as s:
        for i in range(0, total, batch_size):
            s.run("""
                UNWIND $rows AS row
                MATCH (e:Estacion {nombre: row.estacion})
                MERGE (a:Afluencia {fecha: row.fecha, estacion: row.estacion, linea: row.linea})
                SET a.anio      = row.anio,
                    a.mes       = row.mes,
                    a.afluencia = row.afluencia
                MERGE (e)-[:TIENE_AFLUENCIA]->(a)
            """, rows=rows[i:i+batch_size])
            print(f"  {min(i+batch_size, total)}/{total}", end="\r")
    print(f"✓ {total} registros de afluencia importados")